In [2]:
import json
from collections import Counter
data_path = "training_data.jsonl"
with open(data_path) as f:
        training_data = [json.loads(line) for line in f]
print(f"  Loaded {len(training_data)} examples")

task_names = {0: "GSM8K", 1: "IFEval", 2: "Code"}
dist = Counter(ex["task_id"] for ex in training_data)
for tid, name in task_names.items():
    print(f"  {name}: {dist.get(tid, 0)}")

labeled = [(ex["messages"], ex["task_id"]) for ex in training_data]


  Loaded 29999 examples
  GSM8K: 10000
  IFEval: 10000
  Code: 9999


In [3]:
from datasets import load_dataset
import re
from collections import defaultdict

# ── Load test sets ────────────────────────────────────────────
gsm8k_test  = load_dataset("openai/gsm8k", "main", split="test")
humaneval   = load_dataset("openai/openai_humaneval", split="test")
ifeval      = load_dataset("google/IFEval", split="train")

# ── Build fingerprints ────────────────────────────────────────
def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def get_ngrams(text, n=8):
    tokens = normalize(text).split()
    return set(' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

# Exact match sets
gsm8k_questions   = {normalize(ex['question']) for ex in gsm8k_test}
humaneval_prompts = {normalize(ex['prompt'])   for ex in humaneval}
ifeval_prompts    = {normalize(ex['prompt'])   for ex in ifeval}

# N-gram sets for fuzzy matching
gsm8k_ngrams, humaneval_ngrams, ifeval_ngrams = set(), set(), set()
for ex in gsm8k_test:
    gsm8k_ngrams.update(get_ngrams(ex['question']))
for ex in humaneval:
    humaneval_ngrams.update(get_ngrams(ex['prompt']))
for ex in ifeval:
    ifeval_ngrams.update(get_ngrams(ex['prompt']))

print(f"Fingerprints built:")
print(f"  GSM8K test:  {len(gsm8k_questions)} exact | {len(gsm8k_ngrams)} ngrams")
print(f"  HumanEval:   {len(humaneval_prompts)} exact | {len(humaneval_ngrams)} ngrams")
print(f"  IFEval:      {len(ifeval_prompts)} exact | {len(ifeval_ngrams)} ngrams")

# ── Scan training_data_trial ──────────────────────────────────
contaminated = []

for i, ex in enumerate(training_data):
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )
    norm_text = normalize(user_turn)
    example_ngrams = get_ngrams(user_turn)

    # Exact match
    exact_gsm8k    = norm_text in gsm8k_questions
    exact_humaneval = norm_text in humaneval_prompts
    exact_ifeval   = norm_text in ifeval_prompts

    # Fuzzy match (3+ overlapping 8-grams = suspicious)
    gsm8k_overlap    = len(example_ngrams & gsm8k_ngrams)
    humaneval_overlap = len(example_ngrams & humaneval_ngrams)
    ifeval_overlap   = len(example_ngrams & ifeval_ngrams)

    if (exact_gsm8k or exact_humaneval or exact_ifeval or
        gsm8k_overlap >= 3 or humaneval_overlap >= 3 or ifeval_overlap >= 3):
        contaminated.append({
            'index':              i,
            'task_id':            ex.get('task_id'),
            'source':             ex.get('source', 'unknown'),
            'exact_gsm8k':        exact_gsm8k,
            'exact_humaneval':    exact_humaneval,
            'exact_ifeval':       exact_ifeval,
            'gsm8k_overlap':      gsm8k_overlap,
            'humaneval_overlap':  humaneval_overlap,
            'ifeval_overlap':     ifeval_overlap,
            'text_preview':       user_turn[:200],
        })

print(f"\nScanned {len(training_data)} training examples")
print(f"Flagged: {len(contaminated)}")

if contaminated:
    import pandas as pd
    df = pd.DataFrame(contaminated)
    print("\nBreakdown by source:")
    print(df['source'].value_counts())
    print("\nBreakdown by task_id:")
    print(df['task_id'].value_counts())
    print("\nSample flagged examples:")
    for row in contaminated[:3]:
        print(f"\n  [{row['source']}] gsm8k_overlap={row['gsm8k_overlap']} "
              f"he_overlap={row['humaneval_overlap']} if_overlap={row['ifeval_overlap']}")
        print(f"  {row['text_preview']}")

/opt/miniconda3/envs/cps572/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fingerprints built:
  GSM8K test:  1319 exact | 51706 ngrams
  HumanEval:   163 exact | 8730 ngrams
  IFEval:      541 exact | 15184 ngrams

Scanned 29999 training examples
Flagged: 406

Breakdown by source:
source
unknown                                                 253
ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980    153
Name: count, dtype: int64

Breakdown by task_id:
task_id
2    243
1    153
0     10
Name: count, dtype: int64

Sample flagged examples:

  [unknown] gsm8k_overlap=0 he_overlap=3 if_overlap=0
  Read the following function signature and docstring, and fully implement the function described. Your response should only contain the code for this function.
You are given a string `s`. Your task is 

  [unknown] gsm8k_overlap=0 he_overlap=3 if_overlap=0
  Read the following function signature and docstring, and fully implement the function described. Your response should only contain the code for this function.
You are given a string `s`. Your task is 

  [ai2-adapt

In [4]:
# Check exact matches only — these are definitive contamination
exact_contamination = df[
    df['exact_gsm8k'] | df['exact_humaneval'] | df['exact_ifeval']
]
print(f"Exact matches (definitive contamination): {len(exact_contamination)}")

# Check high overlap — more suspicious than threshold=3
high_overlap = df[
    (df['gsm8k_overlap'] >= 6) |
    (df['humaneval_overlap'] >= 6) |
    (df['ifeval_overlap'] >= 6)
]
print(f"High overlap (>=6 ngrams): {len(high_overlap)}")

# Print exact matches for manual inspection
for _, row in exact_contamination.iterrows():
    print(f"\n  source: {row['source']} | task_id: {row['task_id']}")
    print(f"  {row['text_preview']}")

Exact matches (definitive contamination): 0
High overlap (>=6 ngrams): 11


In [7]:
HUMANEVAL_INSTRUCTION = """Read the following function signature and docstring, and fully implement the function described. Your response should only contain the code for this function.\n"""

In [ ]:
# For each high-overlap case, show the training example AND the overlapping test examples

def find_overlapping_test_examples(training_text, test_dataset, text_field, n=8, threshold=6):
    training_ngrams = get_ngrams(training_text)
    overlapping = []
    for test_ex in test_dataset:
        test_text = test_ex[text_field]
        test_ngrams = get_ngrams(test_text)
        overlap = training_ngrams & test_ngrams
        if len(overlap) >= threshold:
            overlapping.append({
                'test_text': test_text,        # full text, no truncation
                'overlap_count': len(overlap),
                'shared_ngrams': list(overlap) # all shared ngrams
            })
    return sorted(overlapping, key=lambda x: -x['overlap_count'])


TRAINING EXAMPLE (index=3649, source=ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980, task_id=1)
Text: Title: Key Considerations for Optimizing Grid Integration of Renewable Energy

Give me a comprehensive overview of key considerations for optimizing grid integration of renewable energy projects. Prov

── IFEval overlaps (overlap=8) ──
  Test prompt: Write a rubric, in the form of a list of bullet points, for evaluating the performance of a customer service representative. Your answer must not include keywords ['bad', 'underperform'] and must contain exactly 6 bullet points in the following form:
* Bullet point 1
* Bullet point 2
* Bullet point 3
* Bullet point 4
* Bullet point 5
* Bullet point 6
  Shared ngrams (8): ['point 2 bullet point 3 bullet point 4', 'bullet point 1 bullet point 2 bullet point', 'bullet point 2 bullet point 3 bullet point', 'point 1 bullet point 2 bullet point 3', 'bullet point 3 bullet point 4 bullet point', '2 bullet point 3 bullet point 4 bullet', 'poi

In [10]:
for _, row in high_overlap.iterrows():
    print("=" * 80)
    
    # Full training text
    ex = training_data[row['index']]
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )
    if ex.get('task_id') == 2:
        user_turn = user_turn.replace(HUMANEVAL_INSTRUCTION, "").strip()

    print(f"TRAINING (index={row['index']}, source={row['source']}, task_id={row['task_id']})")
    print(f"FULL TEXT:\n{user_turn}")
    print()

    # Full overlapping test text
    if row['gsm8k_overlap'] >= 6:
        matches = find_overlapping_test_examples(user_turn, gsm8k_test, 'question', threshold=6)
        for m in matches:
            print(f"── GSM8K TEST (overlap={m['overlap_count']}) ──")
            print(f"FULL TEXT:\n{m['test_text']}")
            print(f"SHARED NGRAMS: {m['shared_ngrams']}")
            print()

    if row['humaneval_overlap'] >= 6:
        matches = find_overlapping_test_examples(user_turn, humaneval, 'prompt', threshold=6)
        for m in matches:
            print(f"── HUMANEVAL TEST (overlap={m['overlap_count']}) ──")
            print(f"FULL TEXT:\n{m['test_text']}")
            print(f"SHARED NGRAMS: {m['shared_ngrams']}")
            print()

    if row['ifeval_overlap'] >= 6:
        matches = find_overlapping_test_examples(user_turn, ifeval, 'prompt', threshold=6)
        for m in matches:
            print(f"── IFEVAL TEST (overlap={m['overlap_count']}) ──")
            print(f"FULL TEXT:\n{m['test_text']}")
            print(f"SHARED NGRAMS: {m['shared_ngrams']}")
            print()

TRAINING (index=3649, source=ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980, task_id=1)
FULL TEXT:
Title: Key Considerations for Optimizing Grid Integration of Renewable Energy

Give me a comprehensive overview of key considerations for optimizing grid integration of renewable energy projects. Provide the overview in a bullet list format with 5 points. Exclude the words "fossil fuels," "nuclear," and "non-renewable" from your response. 

* Bullet Point 1
* Bullet Point 2
* Bullet Point 3
* Bullet Point 4
* Bullet Point 5

── IFEVAL TEST (overlap=8) ──
FULL TEXT:
Write a rubric, in the form of a list of bullet points, for evaluating the performance of a customer service representative. Your answer must not include keywords ['bad', 'underperform'] and must contain exactly 6 bullet points in the following form:
* Bullet point 1
* Bullet point 2
* Bullet point 3
* Bullet point 4
* Bullet point 5
* Bullet point 6
SHARED NGRAMS: ['point 2 bullet point 3 bullet point 4', 'bullet point 1